# 01 · Corpus — articles to shards on S3

**Runs on: the store box** (the one with `kubectl`). No GPU needed.

Pulls every article with a usable abstract out of the platform index, chunks it
with the library's own 512/64 window, and uploads the shards to S3 for the GPU
box to pick up. Also ships the ontology, so the GPU box needs nothing but S3
credentials.

```
  STORE BOX (kubectl)              S3  s3.wisefood-project.eu           GPU BOX
  ─────────────────────            ──────────────────────────           ───────
  01_corpus                                                             02_annotate
    articles ──chunk──────────────> corpus/shard_*.csv  ──────────────>  download
                                    ontology/foodon.*   ──────────────>  download
                                                                         NER + NEL + embed
                                                                         (parallel)
  03_layers                         annotated/shard_*.parquet  <───────  upload
    download <──────────────────────
    ingest → prod Elasticsearch
    Layer A/B/C → prod Neo4j
```

Neither box needs what the other has. The GPU box never reaches Elasticsearch
or Neo4j — they are `ClusterIP` with no ingress and it could not if it wanted
to. The store box never needs a GPU. MinIO is the only thing both can see, and
it already has one: `s3.wisefood-project.eu` resolves publicly and answers
`/minio/health/live`.

`FS_RUN_ID` is the join. Use the same string on both machines; it namespaces
every object under `runs/<FS_RUN_ID>/`, so two builds cannot interleave.

## 0. Preflight

```bash
kubectl --context k8s-w -n wf-prod port-forward svc/elastic 9200:9200 &

export ELASTICSEARCH_URL=http://localhost:9200
export S3_ACCESS_KEY=root
export S3_SECRET_KEY=...          # minio_root secret, key `password`
export FS_RUN_ID=2026-09-21-full  # the SAME string on the GPU box
```

The textbook and guide chunks are **not** touched. They are already annotated
in the platform index with stable ids, and re-chunking assigns fresh UUIDs that
orphan every card citing them. Only the abstracts are new.

In [ ]:
import os, time, json, math
from pathlib import Path

REQUIRED = ["ELASTICSEARCH_URL", "S3_ACCESS_KEY", "S3_SECRET_KEY", "FS_RUN_ID"]
missing = [v for v in REQUIRED if not os.environ.get(v)]
if missing:
    raise SystemExit(f"missing environment: {', '.join(missing)}")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
WORK = DATA / "handoff" / os.environ["FS_RUN_ID"]
WORK.mkdir(parents=True, exist_ok=True)

ES_URL = os.environ["ELASTICSEARCH_URL"]
ARTICLE_INDEX = os.environ.get("FS_ARTICLE_INDEX", "articles")
CHUNK_INDEX = os.environ.get("FS_CHUNK_INDEX", "foodscholar_chunks")
print(ROOT)

In [ ]:
import os, json, hashlib
from pathlib import Path

try:
    import boto3
    from botocore.client import Config as BotoConfig
except ImportError as e:
    raise SystemExit("pip install boto3  # S3 handoff between the GPU box and the store box") from e

S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "https://s3.wisefood-project.eu")
S3_BUCKET   = os.environ.get("S3_BUCKET", "foodscholar-graph-build")
RUN_ID      = os.environ["FS_RUN_ID"]          # same string on both machines

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    # MinIO speaks path-style; virtual-host style would resolve
    # <bucket>.s3.wisefood-project.eu, which has no DNS record.
    config=BotoConfig(signature_version="s3v4", s3={"addressing_style": "path"}),
)

def s3_key(*parts: str) -> str:
    return "/".join(["runs", RUN_ID, *parts])

def s3_exists(key: str) -> bool:
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=key)
        return True
    except Exception:
        return False

def s3_put(local: Path, key: str) -> None:
    s3.upload_file(str(local), S3_BUCKET, key)

def s3_get(key: str, local: Path) -> Path:
    local.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(S3_BUCKET, key, str(local))
    return local

def s3_list(prefix: str) -> list[str]:
    keys, token = [], None
    while True:
        kw = {"Bucket": S3_BUCKET, "Prefix": prefix}
        if token:
            kw["ContinuationToken"] = token
        resp = s3.list_objects_v2(**kw)
        keys.extend(o["Key"] for o in resp.get("Contents", []))
        if not resp.get("IsTruncated"):
            return sorted(keys)
        token = resp["NextContinuationToken"]

print(f"s3  {S3_ENDPOINT}/{S3_BUCKET}")
print(f"run {RUN_ID}")

## 1. What the platform already holds

In [ ]:
from elasticsearch import Elasticsearch
es = Elasticsearch(hosts=ES_URL)

def count(index, body=None):
    return int(es.count(index=index, body=body)["count"]) if es.indices.exists(index=index) else 0

by_source = {}
if es.indices.exists(index=CHUNK_INDEX):
    aggs = es.search(index=CHUNK_INDEX, size=0,
                     body={"aggs": {"s": {"terms": {"field": "source_type", "size": 10}}}})
    by_source = {b["key"]: b["doc_count"] for b in aggs["aggregations"]["s"]["buckets"]}

print("chunk index, by source_type:")
for k, v in sorted(by_source.items()):
    print(f"  {k:10s} {v:7d}")
print(f"articles available: {count(ARTICLE_INDEX)}")

## 2. Chunk the abstracts

`fs.chunk_texts()` — the abstracts path, which has no CLI. Same sliding window
as the PDFs over NLTK sentences rather than Docling units.

Stores are **memory** here: this step writes a CSV and touches no database.
Needs `foodscholar[chunking]` for nltk and the tokenizer.

In [ ]:
from foodscholar import FoodScholar

CHUNK_CONFIG = {
    "corpus": {"chunks_path": str(WORK / "chunks.parquet")},
    "ontology": {"foodon_path": str(DATA / "foodon.owl"),
                 "cache_path": str(DATA / "foodon_cache.parquet")},
    "storage": {"chunk_store": {"backend": "memory"},
                "graph_store": {"backend": "memory"},
                "card_store": {"backend": "memory"}},
}
fs = FoodScholar.from_config(CHUNK_CONFIG)

MIN_ABSTRACT_CHARS = int(os.environ.get("FS_MIN_ABSTRACT_CHARS", "200"))
ARTICLE_LIMIT = int(os.environ.get("FS_ARTICLE_LIMIT", "0"))  # 0 = all; set small to rehearse

texts, meta, after, scanned, skipped = {}, {}, None, 0, 0
while True:
    body = {"size": 1000,
            "_source": ["urn", "id", "title", "abstract", "publication_year", "doi"],
            "query": {"exists": {"field": "abstract"}},
            "sort": [{"_id": "asc"}]}
    if after:
        body["search_after"] = after
    hits = es.search(index=ARTICLE_INDEX, body=body)["hits"]["hits"]
    if not hits:
        break
    for h in hits:
        s = h["_source"]
        scanned += 1
        abstract = (s.get("abstract") or "").strip()
        # A 40-character "abstract" is a placeholder, not evidence. Chunking it
        # makes a node with nothing behind it and inflates support counts.
        if len(abstract) < MIN_ABSTRACT_CHARS:
            skipped += 1
            continue
        doc_id = s.get("urn") or s.get("id") or h["_id"]
        texts[doc_id] = abstract
        meta[doc_id] = {"title": s.get("title") or "",
                        "year": s.get("publication_year") or "",
                        "doi": s.get("doi") or ""}
    after = hits[-1]["sort"]
    if ARTICLE_LIMIT and len(texts) >= ARTICLE_LIMIT:
        break

print(f"scanned {scanned}, kept {len(texts)}, skipped {skipped} short/empty")

In [ ]:
abstracts_csv = WORK / "abstracts.csv"
if abstracts_csv.exists() and abstracts_csv.stat().st_size > 0:
    print(f"reusing {abstracts_csv} — delete to re-chunk")
else:
    t0 = time.perf_counter()
    abstracts_csv = fs.chunk_texts(texts, out_path=abstracts_csv,
                                   source_type="abstract", metadata=meta)
    print(f"wrote {abstracts_csv} in {time.perf_counter()-t0:.0f}s")

from foodscholar.corpus import load_chunks
all_chunks = load_chunks(abstracts_csv)
print(f"{len(all_chunks)} abstract chunks")

## 3. Shard and upload

Shard size is a restart granularity, not a performance knob — a shard is the
unit the GPU box redoes if it dies. 500 chunks is a couple of minutes of GPU
work, which is a cheap thing to lose.

In [ ]:
import csv

SHARD_SIZE = int(os.environ.get("FS_SHARD_SIZE", "500"))
shard_dir = WORK / "corpus"
shard_dir.mkdir(parents=True, exist_ok=True)

rows = list(csv.DictReader(abstracts_csv.open()))
shards = [rows[i:i + SHARD_SIZE] for i in range(0, len(rows), SHARD_SIZE)]
fieldnames = rows[0].keys() if rows else []

manifest = []
for i, shard in enumerate(shards):
    p = shard_dir / f"shard_{i:04d}.csv"
    if not p.exists():
        with p.open("w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=fieldnames)
            w.writeheader()
            w.writerows(shard)
    key = s3_key("corpus", p.name)
    if not s3_exists(key):
        s3_put(p, key)
    manifest.append({"shard": p.name, "rows": len(shard), "key": key})

print(f"{len(shards)} shards uploaded under {s3_key('corpus')}/")

## 4. Ship the ontology too

40 MB of OWL plus its parsed cache. Uploading it means the GPU box provisions
itself from S3 and needs nothing hand-copied.

In [ ]:
for f in ("foodon.owl", "foodon_cache.parquet", "foodon_cache.parquet.meta.json"):
    src = DATA / f
    if not src.exists():
        print(f"  !! missing {src}")
        continue
    key = s3_key("ontology", f)
    if s3_exists(key):
        print(f"  already on s3: {f}")
    else:
        t0 = time.perf_counter()
        s3_put(src, key)
        print(f"  uploaded {f} ({src.stat().st_size/1e6:.1f} MB, {time.perf_counter()-t0:.0f}s)")

manifest_path = WORK / "manifest.json"
manifest_path.write_text(json.dumps(
    {"run_id": RUN_ID, "shard_size": SHARD_SIZE, "shards": manifest,
     "chunks": len(rows), "articles": len(texts)}, indent=2))
s3_put(manifest_path, s3_key("manifest.json"))
print(f"\nmanifest: {len(rows)} chunks in {len(shards)} shards")
print(f"\nNext: run 02_annotate on the GPU box with FS_RUN_ID={RUN_ID}")